# 00. Function (step-by-step)

In [1]:
import pandas as pd
import numpy as np
import re
import os
import sys

#-----
sys.path.append(os.path.abspath("..")) #“From my notebook folder, go up one level and add that directory to Python’s import search path.”
from src import config

PRODUCTION_FILE = config.production_file
VENDOR_FOLDER = config.vendor_folder
DEBUG_FOLDER = config.debug_folder

BIG_M = config.BIG_M
START_DATE = config.START_DATE
END_DATE   = config.END_DATE




In [2]:
print(DEBUG_FOLDER)

C:\Users\David\OneDrive\TAIDUC\0. DATASET. MATERIAL\Optimization\Procurement-Vendor quantity selection\debugs


### supported function

In [3]:
from gurobipy import GRB
def export_gurobi_model(model, name:str, silent: bool = False):
    import os
    from datetime import datetime
    
    if silent:
        return #exit function = Do not write the model
    else:
        # prepare
        timestamp = datetime.now().strftime("%Y%m%d_%H%M")
        model_folder = os.path.join(DEBUG_FOLDER , "models")
        os.makedirs(model_folder, exist_ok=True)
        
        file_name = f"{name}_{timestamp}.lp"
        full_path = os.path.join(model_folder, file_name)
        # write
        model.write(full_path)
        #print(f"[>] Exporting model : {os.path.basename(full_path)}")
    
def apply_gurobi_log_settings(model, name: str, silent: bool = True):
    """
    Configures Gurobi model to save logs to a specific timestamped file.
    
    Args:
        model: The Gurobi model object.
        name: Short description for the log file (e.g., 'BaseModel').
        silent: If True, disables log printing on the terminal.
    """
    
    # FOR VERSION CONTROL:
    # If you are using Git, remember to add 'debugs/logs/*.log' to your .gitignore file.
    # This prevents tracking hundreds of heavy log files and keeps your repository clean!
    import os
    from datetime import datetime
    
    # 1. Setup folders
    log_folder = os.path.join(DEBUG_FOLDER , "logs")
    os.makedirs(log_folder, exist_ok=True)
    
    # 2. Setup path and file name
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    file_name = f"{name}_{timestamp}.log"
    full_path = os.path.join(log_folder, file_name)
    
    # 3. Apply Gurobi Parameters

    if silent:
        # CHOICE A: Hard silence (Best for production performance)
        #model.setParam("OutputFlag", 0) # Disables ALL log generation. Your .log file will be EMPTY.
        
        # CHOICE B: Soft silence (Best for background debugging)
        
        model.setParam("LogToConsole", 0) #Keeps terminal clean but CONTINUES recording details to your .log file.
        
    model.setParam("LogFile", full_path)

        
    
    # model.setParam("DisplayInterval", 1) # optional: detail log for debugging (lv2)
    
    #print(f"[>] Logging status  : {os.path.basename(full_path)}")
    
    # Example Usage:
    # apply_gurobi_log_settings(base_model, "01_BaseModel")
    
    
def get_gurobi_log_path(name: str):
    import os
    from pathlib import Path
    
    log_folder = os.path.join(DEBUG_FOLDER , "logs")
    file_name = f"{name}.log"
    full_path = os.path.join(log_folder, file_name)
    
    
    return f".../debugs/logs/{file_name}"


def analyze_infeasbility_result(relax_model, max_feasible_model , df_relax_options, df_info):
    
    infeasible_items_row = df_info.loc[df_info['Metric'].str.contains("Infeasible Items", na=False)]
    infeasible_vendors_row = df_info.loc[df_info['Metric'].str.contains("Infeasible Vendors", na=False)]
    try:
        num_infeasible_items = infeasible_items_row['Value'].values[0]
        num_infeasible_vendors = infeasible_vendors_row['Value'].values[0]
    except IndexError:
    # Phòng trường hợp không tìm thấy hàng nào
        num_infeasible_items = 0
        num_infeasible_vendors = 0
    
    
    print("[i]   MAX FEASIBLE SUMMARY:")
    print(f"     - Total Current Cost                      : ${max_feasible_model.ObjVal:.2f}")
    print(f"     - Infeasible Items (Supply Gap Violation) : {num_infeasible_items}")
    print(f"     - Infeasible Vendors (Revenue Violation)  : {num_infeasible_vendors}")
    print("[i]   RELAXATION SUMMARY:")
    constr = relax_model.getConstrByName("feasobj")
    rhs = constr.RHS if constr is not None else 0
    if rhs is None:
        penalty_cost = relax_model.ObjVal #or for c in model.getConstrs() --> #if "feasobj" in c.ConstrName --> print(c.ConstrName, c.RHS)
    else:
        penalty_cost = rhs
    original_obj = sum(v.Obj * v.X for v in relax_model.getVars()) + relax_model.ObjCon
    
    violation_count = sum(1 for v in relax_model.getVars() if "Art" in v.VarName and abs(v.X) > 1e-6)
    
    print(f"     - Total Penalty Cost   : {penalty_cost:.2f}")
    print(f"     - Total Target Cost : ${original_obj:.2f}")
    print(f"     - Total Relaxation Count: {violation_count}")
    
    print("[i]   ADJUSTMENTS NEEDED:")
    formatted_list = [
    f"- {row['Phân_Loại']} | {row['Sản_Phẩm']} | {row['Nhà_Cung_Cấp']} | Brk {row['Khung_Giá_Bracket']} : Delay +{str(row['Kế_Hoạch_Thực_Tế']).replace(' ngày', '')} days"
    for _, row in df_relax_options.iterrows()
    ]
    
    n = len(formatted_list)

    if n == 0:
        print("No records found.")
    else:
        # Luôn in các dòng hiện có, tối đa là 2
        for line in formatted_list[:2]:
            print(line)
        
        # Chỉ in dòng "and others" nếu thực sự còn dư
        if n > 2:
            print(f"... (and {n - 2} others)")
    
    
def print_header():
    from datetime import datetime
    time_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    border = "=" * 78
    print(f"\n{border}")
    print(f"procurement-optimizer v2.0 | {time_str}")
    print(f"{border}\n")

def print_final_summary(base_model):
    
    if base_model.Status == GRB.Status.INFEASIBLE:
        status_text = "INFEASIBLE (Relaxed Solution Provided)"
        path_file = "...output_INFEASIBILITY.xlsx"
    else:
        status_text = "FEASIBLE"
        path_file = "...output_FEASIBILITY.xlsx"
    
    path_log = ".../debugs/logs/"
    
    
    
    print("") 
    print("=" * 78) 
    print("✅ PROCESS COMPLETED")
    print("=" * 78) 
    print(f"{'[FINAL STATUS]':<15}: {status_text}")
    print(f"{'[OUTPUT FILE]':<15}: {path_file}")
    print(f"{'[LOG FOLDER]':<15}: {path_log}")
    print("=" * 78)
    
    

## 01. prepare input

### raw --> processed data

In [4]:
def processing_demand(data: pd.DataFrame) -> pd.DataFrame:
    
    print(f"[>] Aggregating demand by item & timeline...", end="\r") #end="\r" (The Spinner Effect):
    
    
    try:
        data['days_until_demand'] = (data['delivery_date'] - START_DATE).dt.days
        data['min_days_until_demand'] = (
                                                data.groupby('item_code')['days_until_demand']
                                                .transform('min')
                                            )

        grouped_data = (                             
                data
                .groupby(['item_code','min_days_until_demand'], as_index=False) #, 'days_until_demand'
                .agg(req_qty=('required_qty', 'sum'))
    )


        print(f"[OK] Demand Aggregation: {len(grouped_data)} unique item-groups.")
        return grouped_data
    except Exception as e:
        
        print(f"[!] Error in processing_demand: {e}")
        raise
    
    



### processed data --> model Input {Sets & Constants}

In [5]:

def prepare_model_input(raw_demand_grouped, raw_supply):
    """
    - input: raw datasets & 
    - output: Constants & Parameter 
    
    """
    
    
    try:
        # --------------------------------------------------------------------------------------------------------
        demand = raw_demand_grouped.copy() #demand['req_qty'] = demand['req_qty'] * 3

        capacity = raw_supply['capacity'] #capacity['capacity'] = 500
        leadtime = raw_supply['leadtime']
        price = raw_supply['price']
        revenue = raw_supply['revenue'] #revenue['revenue'] = revenue['revenue'] + 450000

            # scenario -----------------
        new_demand = demand.copy()
        new_demand['req_qty'] = new_demand['req_qty'] * 2

        new_capacty = capacity.copy() #adjust for testing
        new_capacty.loc[new_capacty["item_code"] == "ITM004","capacity"] += 1000
        new_revenue = revenue.copy()
        new_revenue['revenue'] = new_revenue['revenue'] + 450000

        # --------------------------------------------------------------------------------------------------------

        # -------------------- SETS ------------------------
        IDX = list(price.set_index(['item_code','vendor','bracket_no']).index) #for (i,s,b) in IDX:
        I = price['item_code'].unique().tolist() #set of item codes
        J = price['vendor'].unique().tolist() #set of vendors
        K = price['bracket_no'].unique().tolist()#set of brackets
            # Sparse sets: items per vendor, vendors per item , Đây là key addition — tránh cross-join vô nghĩa
        I_s = price.groupby('vendor')['item_code'].apply(set).to_dict()   # {vendor: {items}}
        J_m = price.groupby('item_code')['vendor'].apply(set).to_dict()   # {item: {vendors}}
        IJ_b = price.groupby(['item_code', 'vendor'])['bracket_no'].apply(list).to_dict() # {(vendor-item) : [bracket_list] }
        IJ_pair = set((i,v) for (i,v,b) in IDX )
            # from collections import defaultdict
            # IJ_b = defaultdict(list)
            # for i, v, b in IDX:
            #     IJ_b[(i, v)].append(b)

        # -------------------- CONSTANTS / PARAMETERS ------------------------

        D = demand.set_index(['item_code'])['req_qty'].to_dict() #set of demand

        MD = demand.set_index(['item_code'])['min_days_until_demand'].to_dict() #mins day
        C = capacity.set_index(['item_code','vendor'])['capacity'].to_dict() #set of capacity 
        L = leadtime.set_index(['item_code','vendor','bracket_no'])['lead_time'].to_dict() #set of leadtime 
        P = price.set_index(['item_code','vendor','bracket_no'])['price'].to_dict() #set of price
        R = revenue.set_index(['vendor'])['revenue'].to_dict() #set of revenue 

        bounds = leadtime.set_index(['item_code','vendor','bracket_no'])[['lower_bound','upper_bound']]
        LB = bounds['lower_bound'].to_dict()
        UB = bounds['upper_bound'].to_dict()
        
        
        result = {
        #----Obj index
            "index": IDX,
        #----SETS
            "item_code": I,
            "vendor":J,
            "bracket":K,
            "vendor_item": I_s,
            "vendorItem_bracket": IJ_b,
            "vendorItem_pair": IJ_pair,
            "item_vendor": J_m,
        #----PARAMETER
            "demand_data": D,
            "minRequiredDate_data": MD,
            "capacity_data": C,
            "leadtime_data": L,
            "price_data": P,
            "revenue_data":R,
            "lowerBound_data": LB,
            "upperBound_data": UB
        }
        
        
        # -------------------- FINAL SUCCESS LOG ------------------------
        print(f"[OK] Model Input Ready:")
        print(f"       [+] Items    : {len(I)}")
        print(f"       [+] Vendors  : {len(J)}")
        print(f"       [+] Brackets : {len(K)}")
        print(f"       [+] Indices  : {len(IDX):,} (Sparse Pairs: {len(IJ_pair)})")
        
        return result
    
    
    
    except Exception as e:
        print(f"[!] Error in prepare_model_input: {e}")
        raise
    


    

## 02. build & solve model

### base model

In [6]:
import pandas as pd
import numpy as np

import gurobipy as gp
from gurobipy import GRB


def solve_model_main(input_model):
    
    # ------- extract input --------- 
    IDX  = input_model["index"]
    I    = input_model["item_code"]
    J    = input_model["vendor"]
    K    = input_model["bracket"]
    I_s  = input_model["vendor_item"]
    J_m  = input_model["item_vendor"]
    D    = input_model["demand_data"]
    MD   = input_model["minRequiredDate_data"]
    C    = input_model["capacity_data"]
    L    = input_model["leadtime_data"]
    P    = input_model["price_data"]
    R    = input_model["revenue_data"]
    LB   = input_model["lowerBound_data"]
    UB   = input_model["upperBound_data"]
    
    # ------  Model INITIALIZATION ---------  

    model = gp.Model("supplier_allocation")

    # ------  Decision Variables  ---------  

    x = model.addVars(IDX, vtype= GRB.CONTINUOUS, name = "buying_qty", lb=0)
    y = model.addVars(IDX, vtype= GRB.BINARY, name = "bracket_selection")

    # ------  Objective function: ---------

    model.setObjective( 
            gp.quicksum(P[m,s,b] * x[m,s,b] for (m,s,b) in IDX),
            GRB.MINIMIZE
        
    )


    # ------  CONSTRAINT ---------
    # ----- BUSINESS
    # 1. Demand Fulfillment 
    for m in I:
        model.addConstr(	gp.quicksum(x[m,s,b] for (mm,s,b) in IDX if mm==m ) >= D[m],
            name = f"demand_{m}"
        )
    # 2. Supplier capacity

    for m in I:
        for s in J_m[m]:
            model.addConstr(
                gp.quicksum(x[m,s,b] for (mm,ss,b) in IDX if mm==m and ss==s) <= C[m,s],
                name = f"capacity_{m}_{s}"
            )
            
    # 3. Revenue commitment
    for s in J:
        model.addConstr(
            gp.quicksum(x[m,s,b] * P[m,s,b] for (m,ss,b) in IDX if ss==s ) >= R[s],
            name = f"revenue_{s}"
        )
        
    # 4. Leadtime feasibility

    F = {
        (m,s,b): 1 if L[m,s,b] <= MD[m] else 0
        for (m,s,b) in IDX
    }


    for (m,s,b) in IDX:
        model.addConstr(
            y[m,s,b] <= F[m,s,b],
            name=f"leadTimeFeasible_{m}_{s}_{b}"
        )

    # ----- MATH
    # 5. Bracket Activation = LINKING CONSTRAINT

    for (m,s,b) in IDX:
            model.addConstr(
            x[m,s,b]  >= LB[m,s,b] * y[m,s,b],
            name = f"lowerLimit_{m}_{s}_{b}"
        )
    for (m,s,b) in IDX:
            model.addConstr(
            x[m,s,b]  <= UB[m,s,b] * y[m,s,b],
            name = f"upperLimit_{m}_{s}_{b}"
        )


    # 6. Single Bracket Selection

    for m in I:
        for s in J_m[m]:
            model.addConstr(
                gp.quicksum(y[m,s,b] for (mm,ss,b) in IDX if mm==m and ss==s ) <=1,
                name = f"singleBracket_{m}_{s}"
            )

    # optional. FIXING variable for checking
    FIXING_VARIABLE = False
    if FIXING_VARIABLE == True:
        print("\n\n ============== STARTING FIXING VARIABLES =================== ")
        target_m = "ITM001"
        target_s = "vendor_1"
        target_b = 3  # Bracket số 3
        
        #Ép mua đúng 1500
        x[target_m,target_s,target_b].LB = 1000
        x[target_m,target_s,target_b].UB = 1000
        
        y[target_m, target_s, target_b].LB = 1
        
        y[target_m, target_s, target_b].UB = 1
        
        for b in [1, 2]: # Giả sử có bracket 1 và 2
            y[target_m, target_s, b].UB = 0
        model.update()


    #--------------------------------------
    export_gurobi_model(model, "01_baseline") #save mo hinh
    apply_gurobi_log_settings(model, "01_baseline")
    model.optimize()
    
    return model



### relaxation (infeasibility)

In [7]:
def solve_with_relaxation(model, input_model,debug_model = False):
    # ------- extract input --------- 
    IDX  = input_model["index"]
    I    = input_model["item_code"]
    J    = input_model["vendor"]
    K    = input_model["bracket"]
    I_s  = input_model["vendor_item"]
    J_m  = input_model["item_vendor"]
    D    = input_model["demand_data"]
    MD   = input_model["minRequiredDate_data"]
    C    = input_model["capacity_data"]
    L    = input_model["leadtime_data"]
    P    = input_model["price_data"]
    R    = input_model["revenue_data"]
    LB   = input_model["lowerBound_data"]
    UB   = input_model["upperBound_data"]

    
    # ============= 1. Tạo dictionary chứa Penalty cho từng ràng buộc =============
    
    # Mặc định penalty = None (nghĩa là KHÔNG cho phép nới lỏng)
    constrs = model.getConstrs()
    
    rhspen = []
    
    
    for c in constrs:
        name = c.ConstrName
        if "demand" in name:
            rhspen.append(2000.0)  # Nới lỏng nhu cầu base : 2000 {normal, nhưng demand x3 thì lộ}, 10000 {solve extreme case demand x3, capa 100}
        elif "capacity" in name:
            rhspen.append(1000.0)  # Nới lỏng công suất (hạn chế tối đa)
        elif "revenue" in name:
            rhspen.append(50)  # Nới lỏng cam kết doanh thu
        elif "leadTimeFeasible" in name:
            rhspen.append(10.0)   # Nới lỏng thời gian giao hàng
        elif "lowerLimit" in name or "upperLimit" in name:
            rhspen.append(1500)   # Nới lỏng khoảng số lượng (Bracket Bounds)
        else:  # các constraint còn lại không được động tới
            rhspen.append(float("inf"))
        
            

    #  ============= 2. Thực hiện FeasRelax =============
    
    # Tham số 0: Tối thiểu hóa tổng độ vi phạm (L1 norm)
    # Tham số True: Tìm phương án vi phạm ít nhất có thể
    model.feasRelax(
        relaxobjtype=0,   # L1 norm
        minrelax=True,
        vars= None,
        lbpen=None,
        ubpen=None,
        constrs= constrs,
        rhspen=rhspen # constraint-level control
    )

    export_gurobi_model(model, "02_Relaxation") #save mo hinh
    apply_gurobi_log_settings(model, "02_Relaxation")
    model.optimize()
        
    # ============= 3. Phân tích kết quả sau nới lỏng =============
    if model.Status in [GRB.OPTIMAL, GRB.SUBOPTIMAL]:
        print("     [OK] Relaxation solved. Log: .../debugs/logs/02_Relaxation.log")
        
        if debug_model == True:
            print("\n\n--- KẾT QUẢ PHÂN TÍCH NỚI LỎNG ---")
            
            constr = model.getConstrByName("feasobj")
            rhs = constr.RHS if constr is not None else 0
            if rhs is None:
                penalty_cost = model.ObjVal #or for c in model.getConstrs() --> #if "feasobj" in c.ConstrName --> print(c.ConstrName, c.RHS)
            else:
                penalty_cost = rhs
            print(f"Tổng chi phí phạt vi phạm: {penalty_cost:.2f}")
            original_obj = sum(v.Obj * v.X for v in model.getVars()) + model.ObjCon
            print(f"Tổng Chi phí gốc của phương án này: {original_obj:.2f}")

            print("\n--- CHI TIẾT VI PHẠM ---")
            print("\n=== DECISION VARIABLES in RELAXED MODEL ===")
            for v in model.getVars():
                if "Art" in v.VarName and abs(v.X) > 1e-6 : #and abs(v.X) > 1e-6
                    print(v.VarName, v.X)
                    
                # Log riêng cho Leadtime
            for v in model.getVars():
                if "leadTimeFeasible" in v.VarName and v.X > 0.5:
                    print("\n--- [LEADTIME_RELAXATION]: đề xuất dời leadtime ---")
                    print(f"{'Item':<10} | {'Vendor':<12} | {'Bracket':<8} | {'Leadtime':<10} | {'Yêu cầu':<10} | {'Cần lùi':<10}")
                    print("-" * 80)
                    
                    parts = v.VarName.split('_')
                    # Lưu ý: bạn cần kiểm tra index của parts dựa trên cách bạn đặt name trong addConstr
                    # Giả sử: [ArtN, leadTimeFeasible, ItemCode, Vendor, Bracket]
                    m_code = parts[2]
                    s_code = f"{parts[3]}_{parts[4]}" # ghép vendor_1
                    b_no = int(parts[5])

                    # Tính toán số ngày chênh lệch
                    actual_lt = L[m_code, s_code, b_no]
                    required_days = MD[m_code]
                    days_to_extend = actual_lt - required_days

                    print(f"{m_code:<10} | {s_code:<12} | {b_no:<8} | {actual_lt:<10} | {required_days:<10} | +{days_to_extend} ngày")
        
            print("\n=== CONSTRAINT RHS in RELAXED MODEL ===")	
            for c in model.getConstrs():
                #if "Art" in c.ConstrName: #and abs(c.RHS) > 1e-6
                print(c.ConstrName, c.RHS, c.Slack)

    else:
       print("     [ERROR] Relaxation Infeasible. Log: .../debugs/logs/02_Relaxation.log")
        
    return model



### max feasible (infeasibility)

In [8]:
def solve_with_maxFeasible(model, debug_mode = False):
    

    
    # ============= Loại bỏ constraint Revenue =============
    rev_constrs = [c for c in model.getConstrs() if "revenue" in c.ConstrName.lower()]
    for c in rev_constrs:
        model.remove(c)
    model.update()
    
    # ============= 1. Tạo dictionary chứa Penalty cho từng ràng buộc =============
    
    # Mặc định penalty = None (nghĩa là KHÔNG cho phép nới lỏng)
    constrs = model.getConstrs()
    
    rhspen = []
    
    
    for c in constrs:
        name = c.ConstrName
        if "demand" in name:
            rhspen.append(1)  # Nới lỏng nhu cầu
        else:  # các constraint còn lại không được động tới
            rhspen.append(float("inf"))

    #  ============= 2. Thực hiện FeasRelax =============
    
    # Tham số 0: Tối thiểu hóa tổng độ vi phạm (L1 norm)
    # Tham số True: Tìm phương án vi phạm ít nhất có thể
    model.feasRelax(
        relaxobjtype=0,   # L1 norm
        minrelax=True,
        vars= None,
        lbpen=None,
        ubpen=None,
        constrs= constrs,
        rhspen=rhspen # constraint-level control
    )
    export_gurobi_model(model, "03_MaxFeasible") #save mo hinh, or #relax_model.write("relax_model.lp") #save mô hình
    apply_gurobi_log_settings(model, "03_MaxFeasible")
    
    model.optimize()
        
    # ============= 3. Phân tích kết quả sau nới lỏng =============

    if model.Status in [GRB.OPTIMAL, GRB.SUBOPTIMAL]:
        print("     [OK] Relaxation solved. Log: .../debugs/logs/03_MaxFeasible.log")
        
        if debug_mode == True:
            
            print("\n\n--- KẾT QUẢ PHÂN TÍCH NỚI LỎNG ---")
            
            constr = model.getConstrByName("feasobj")
            rhs = constr.RHS if constr is not None else 0
            if rhs is None:
                penalty_cost = model.ObjVal #or for c in model.getConstrs() --> #if "feasobj" in c.ConstrName --> print(c.ConstrName, c.RHS)
            else:
                penalty_cost = rhs

            print(f"Tổng chi phí phạt vi phạm: {penalty_cost:.2f}")
            original_obj = sum(v.Obj * v.X for v in model.getVars()) + model.ObjCon
            print(f"Tổng Chi phí gốc của phương án này: {original_obj:.2f}")
            
            print("\n--- CHI TIẾT VI PHẠM ---")
            print("\n=== DECISION VARIABLES in RELAXED MODEL ===")
            for v in model.getVars():
                if "Art" in v.VarName and abs(v.X) > 1e-6 : #and abs(v.X) > 1e-6
                    print(v.VarName, v.X)
        
            # print("\n=== CONSTRAINT RHS in RELAXED MODEL ===")	
            # for c in model.getConstrs():
            #     #if "Art" in c.ConstrName: #and abs(c.RHS) > 1e-6
            #     print(c.ConstrName, c.RHS, c.Slack)

    else:
        print("     [ERROR] Relaxation Infeasible. Log: .../debugs/logs/03_MaxFeasible.log")
    return model    


## 03. format result

- what: parse results from model output to excel-ready format

### Feasibility (bài toán có nghiệm)

#### Sheet Solutions: Pivot Decision + Detail Decision + Summary

In [9]:
def build_model_solution_outputs(model, raw_supply, input_model):
    """_summary_
    Task: Extract solutions from model , then format into readable results
    Args:
        model (_type_): _description_
        raw_supply (_type_): _description_
        input_model (_type_): _description_

    Returns:
        _type_: _description_
    """
    
    # ------- extract input --------- 
    
    price    = raw_supply['price']
    leadtime = raw_supply['leadtime']
    MD       = input_model["minRequiredDate_data"]
    
    
    # 01. extract solution output
    
    import pandas as pd
    import re
    pattern = re.compile(r"(.+?)\[(.*?),(.*?),(.*?)\]")

    rows = []
    all_rows = []
    for v in model.getVars():
        match = pattern.match(v.VarName)
        if match:
            var_type, item, vendor, bracket = match.groups()
        
            record = {
                "type": var_type.strip(),
                "item_code": item,
                "vendor": vendor,
                "bracket_no": int(bracket),
                "value": v.X
            }
            
            all_rows.append(record)

            if abs(v.X) > 1e-6:
                rows.append(record)

    df = pd.DataFrame(rows if rows else all_rows) # Fallback logic (explicit)
    df_buy = df[df["type"] == "buying_qty"]
    df_bracket = df[df["type"] == "bracket_selection"]
    
    # 02. transform solution output
    
    
    df_buy_details = (
        df_buy
        .rename(columns = {'value': 'qty'})
        .merge(
            price[['vendor', 'item_code', 'bracket_no', 'price','bracket_range']],
            on = ['vendor', 'item_code', 'bracket_no'],
            how = 'left'
        )
        .merge(
            leadtime[['vendor', 'item_code', 'bracket_no', 'lead_time']],
            on = ['vendor', 'item_code', 'bracket_no'],
            how = 'left'
        )
    )
    df_buy_details['required_days'] = df_buy_details['item_code'].map(MD)
    df_buy_details['total_price'] = df_buy_details['qty'] * df_buy_details['price']
    df_buy_details['qty'] = df_buy_details['qty'].round(0).astype(int) #formating
    df_buy_details['total_price'] = df_buy_details['total_price'].round(0).astype(int) #formating
    df_buy_details = df_buy_details[
        ['item_code', 'vendor', 'qty', 'price', 'total_price',
        'bracket_range', 'lead_time', 'required_days']
    ].rename(columns={'price': 'unit price'})
    
    df_buy_summary = (
        df_buy_details
        .fillna({'total_price': 0, 'qty': 0}) #handling missing values
        .groupby('vendor', as_index=False)
        .agg(
            total_allocated_cost=('total_price', 'sum'),
            total_qty=('qty', 'sum')
        )
    )

    pivot_buy = df_buy.pivot_table(
        index=["item_code", "vendor"],
        columns="bracket_no",
        values="value",
        aggfunc="sum",
        fill_value=0
    )
    pivot_bracket = df_bracket.pivot_table(
        index=["item_code", "vendor"],
        columns="bracket_no",
        values="value",
        aggfunc="max",
        fill_value=0
    )
    
    # ----- RESULT
    
    result = {
        "buy_details": df_buy_details,
        "buy_summary": df_buy_summary,
        "pivot_buy": pivot_buy,
        "pivot_bracket": pivot_bracket,
        
    }
    return result



#### Sheet Constraints checking : SupplyDemand, CapacityUsage, Revenue

In [10]:
def build_model_constraint_check(df_buy_details, raw_demand_grouped, raw_supply):
    # --- extract input
    
    demand = raw_demand_grouped.copy()
    capacity = raw_supply['capacity'].copy()
    revenue = raw_supply['revenue'].copy()
    
    # 01. checking Demand vs Supply(total allocated)

    df_buy_by_item = df_buy_details.groupby('item_code', as_index=False).agg(total_allocated = ('qty','sum'))
    df_supplyDemand_check = (
        demand
        .rename(columns = {'req_qty':'total_demand'})
        .merge(
            df_buy_by_item,
            on = 'item_code',
            how = 'left'
        )
        .fillna({'total_allocated': 0}) #handling missing values
    )
    df_supplyDemand_check['gap'] = df_supplyDemand_check['total_allocated'] - df_supplyDemand_check['total_demand']
    df_supplyDemand_check['status'] = np.where(df_supplyDemand_check['gap'] >= 0, 'OK', "SHORTAGE")

    df_supplyDemand_check = df_supplyDemand_check.drop(
        columns=['min_days_until_demand'],
        errors='ignore'
    )

    # 02. checking Capacity Usage

    df_buy_by_vendorItem = df_buy_details.groupby(['vendor','item_code'], as_index=False).agg(total_allocated = ('qty','sum'))
    capacity_summary = capacity.groupby(['vendor','item_code'], as_index=False).agg(total_capacity = ('capacity','sum'))

    df_capacityUsage_Check = (
        capacity_summary
        .merge(
            df_buy_by_vendorItem,
            on = ['item_code', 'vendor'],
            how = 'left'
        )
        .fillna({'total_capacity': 0, 'total_allocated': 0}) #handling missing values
    )

    df_capacityUsage_Check['total_allocated'] = df_capacityUsage_Check['total_allocated'].round(0).astype(int)
    df_capacityUsage_Check['gap'] = df_capacityUsage_Check['total_capacity'] - df_capacityUsage_Check['total_allocated']
    df_capacityUsage_Check['status'] = np.where(df_capacityUsage_Check['gap'] >= 0, 'OK', "SHORTAGE")
    
    df_capacityUsage_Summary_Check = (
        df_capacityUsage_Check.groupby("item_code", as_index=False)
        .agg({
            "total_capacity": "sum",
            "total_allocated": "sum"
        })
    )
    df_capacityUsage_Summary_Check["gap"] = df_capacityUsage_Summary_Check["total_capacity"] - df_capacityUsage_Summary_Check["total_allocated"]
    df_capacityUsage_Summary_Check["status"] = df_capacityUsage_Summary_Check["gap"].apply(lambda x: "OK" if x >= 0 else "SHORT")
    # 03. checking Revenue

    df_revenue_by_vendor = (
        df_buy_details
        .fillna({'total_price': 0, 'qty': 0}) #handling missing values
        .groupby('vendor', as_index=False)
        .agg(
            revenue_allocated=('total_price', 'sum'),
        )
    )

    df_revenueCommit_check = (
        revenue
        .rename(columns = {'revenue': "revenue_commitment"})
        .merge(
            df_revenue_by_vendor,
            on = 'vendor',
            how = 'left'
        )
        .fillna({'revenue_commitment': 0, 'revenue_allocated': 0}) #handling missing values
        
    )

    df_revenueCommit_check['gap'] = df_revenueCommit_check['revenue_allocated'] - df_revenueCommit_check['revenue_commitment']
    df_revenueCommit_check['status'] = np.where(df_revenueCommit_check['gap'] >= 0, 'OK', "Unachieved")

    df_revenueCommit_check
    
    

    result = {
        "supply_demand": df_supplyDemand_check,
        "capacity_usage": df_capacityUsage_Check,
        "capacity_usage_summary": df_capacityUsage_Summary_Check,
        "revenue_commitment": df_revenueCommit_check,
    }
    return result

#### Sheet Model Info (Summary)

In [11]:
def build_model_info(model):
    # 1. Trạng thái giải (Status)
    status_mapping = {
        1: "LOADED", 2: "TỐI ƯU", 3: "INFEASIBLE", 
        4: "INF_OR_UNBD", 5: "UNBOUNDED", 9: "TIME_LIMIT"
    }
    current_status = status_mapping.get(model.Status, f"Status Code: {model.Status}")

    # 2. Đếm số lượng biến theo loại
    # Cách chính xác nhất là duyệt qua danh sách biến
    all_vars = model.getVars()
    count_x = sum(1 for v in all_vars if "buying_qty" in v.VarName) # Thay bằng prefix của bạn
    count_y = sum(1 for v in all_vars if "bracket_selection" in v.VarName) # Thay bằng prefix của bạn
    
    # 3. Tạo list dữ liệu
    info_data = [
        ["Tổng chi phí ($)", f"${model.ObjVal:.0f}"],
        ["Tổng số biến quyết định", model.NumVars],
        ["  - Biến số lượng (x)", count_x],
        ["  - Biến chọn dải (y)", count_y],
        ["Số ràng buộc (Constraints)", model.NumConstrs],
        ["Thời gian giải", f"{model.Runtime:.2f}s"],
        ["Trạng thái", current_status]
    ]

    # 4. Tạo DataFrame
    df_info = pd.DataFrame(info_data, columns=["Metric", "Value"])
    return df_info





### Infeasibility (bài toán vô nghiệm)

phân tích gỡ rối mô hình hiện tại 
- status: 
    - before running FeasRelaxation Model
- what:
    - chạy mô hình MaxFeasible để phân tích mức độ violation và source of "INFEASIBLE" của từng constraints
    - sau đó chạy mô hình FeasRelax (gỡ rối) để đưa ra đề xuất

#### sheet 01_SupplyDemandGap_**ANALYSIS** |||| feed vào sheet 00

In [12]:
def analyze_supply_demand_gap(df_buy_details, raw_demand_grouped, raw_supply):
    """
    Analyzes the supply-demand gap and severity for infeasible models.
    
    This function evaluates:
    - How much supply is available vs how much is demanded
    - The gap percentage for each item
    - The feasibility status (FEASIBLE/INFEASIBLE)
    - The severity level of the shortage (OK/LOW/HIGH/CRITICAL)
    
    INPUT:
        - df_buy_details: Solution data from the MaxFeasible model (contains qty purchased)
        - raw_demand_grouped: Processed demand data (required quantities by item)
        - raw_supply: Raw supply data (contains capacity constraints)
    
    OUTPUT:
        - DataFrame with supply-demand analysis and severity classification
    """
    
    # --- STEP 1: Extract and prepare input data ---
    df_buy_details_TEMPT = df_buy_details.copy()
    demand = raw_demand_grouped.copy()
    capacity = raw_supply['capacity']
    
    # --- STEP 2: Aggregate supply and capacity by item ---
        # Calculate max feasible supply: total quantity that can be allocated across all vendors
    df_buy_by_item = df_buy_details_TEMPT.groupby('item_code', as_index=False).agg(max_feasible_supply = ('qty','sum'))
    
        # Calculate total capacity: total capacity available across all vendors for each item
    capacity_by_item = capacity.groupby('item_code', as_index=False).agg(total_capacity = ('capacity','sum'))
    
    # --- STEP 3: Build comprehensive supply-demand comparison ---
    df_supplyDemand_check = (
        demand
        .rename(columns = {'req_qty':'total_demand'})  # Rename for clarity
        .drop(
        columns=['min_days_until_demand'],
        errors='ignore'
        )
        .merge(capacity_by_item,                        # Add capacity data
            on = 'item_code',
            how = 'left'
        )
        .merge(
            df_buy_by_item,                            # Add feasible supply data
            on = 'item_code',
            how = 'left'
        )
        .fillna({'max_feasible_supply': 0})            # Handle missing values (no supply)
    )
    
    # --- STEP 4: Calculate gap metrics ---

    df_supplyDemand_check['supply_gap'] = df_supplyDemand_check['max_feasible_supply'] - df_supplyDemand_check['total_demand']
    df_supplyDemand_check['gap_percentage'] = (df_supplyDemand_check['supply_gap'] / df_supplyDemand_check['total_demand']) * 100
    df_supplyDemand_check['gap_percentage'] = df_supplyDemand_check['gap_percentage'].round(2)

    # --- STEP 5: Classify feasibility status ---
    import numpy as np
    # If supply_gap >= 0: all demand can be met (FEASIBLE), otherwise INFEASIBLE
    df_supplyDemand_check['feasibility'] = np.where(
        df_supplyDemand_check['supply_gap'] >= 0,
        'FEASIBLE',
        'INFEASIBLE'
    )
    
    # --- STEP 6: Classify severity level based on gap percentage ---
        # Severity classification criteria:
        # - OK: gap >= 0% (supply meets or exceeds demand)
        # - LOW: -5% < gap < 0% (minor shortage, 0-5% shortfall)
        # - HIGH: -20% < gap <= -5% (moderate shortage, 5-20% shortfall)
        # - CRITICAL: gap <= -20% (severe shortage, >20% shortfall)
    conditions = [
        df_supplyDemand_check['gap_percentage'] >= 0,                                                   # OK
        (df_supplyDemand_check['gap_percentage'] >= -5) & (df_supplyDemand_check['gap_percentage'] < 0),  # LOW
        (df_supplyDemand_check['gap_percentage'] >= -20) & (df_supplyDemand_check['gap_percentage'] < -5), # HIGH
        df_supplyDemand_check['gap_percentage'] < -20                                                   # CRITICAL
    ]

    choices = [
        'OK',
        'LOW',
        'HIGH',
        'CRITICAL'
    ]

    df_supplyDemand_check['severity'] = np.select(conditions, choices, default='UNKNOWN')
    
    # Return the analysis dataframe with all metrics
    return df_supplyDemand_check



#### sheet 02_Detail_Items_**ANALYSIS** |||| feed vào sheet 00

In [13]:
def analyze_supply_demand_gap_detail_items(raw_demand, input_model, raw_supply, df_supplyDemand_check):
    """
    Analyzes supply-demand gaps for detail items, considering vendor constraints, lead times, and capacity.
    Parameters:
        raw_demand (DataFrame): Raw demand data with item codes, delivery dates, required quantities, and days until demand.
        input_model (dict): Model inputs including item-vendor mappings and vendor-item brackets.
        raw_supply (dict): Supply data including lead times ,revenue, capacity.
        df_supplyDemand_check (DataFrame): Item-level supply-demand check results from previous analysis (fromm MaxFeas model)
    Returns:
        DataFrame: Detailed analysis with feasibility flags, priority scores, reasons, and diagnostics for each item-delivery date.
    """
    # ------------ STEP 0: Extract input data ------------
    
    J_m = input_model['item_vendor'] #item list of that specific vendor ,, sparse data {item: {vendors}}
    IJ_b = input_model['vendorItem_bracket'] #bracket list of that specific item-vendor ,, sparse data || {(vendor-item) : [bracket_list] }
    leadtime = raw_supply['leadtime']
    capacity = raw_supply['capacity']
    
    detail_demand = raw_demand.copy()
    df_supply_demand_check_ITEMLEVEL = df_supplyDemand_check.copy() #item level, data from analyze_supply_demand_gap function
    

    
    # ------------ STEP 1: formatting input data ------------
    # df_supply_demand_check_ITEMLEVEL
    df_supply_demand_check_ITEMLEVEL['raw_priority_score'] = (
        df_supply_demand_check_ITEMLEVEL['gap_percentage']
        .where(df_supply_demand_check_ITEMLEVEL['feasibility'] == 'INFEASIBLE', 0)
        .mul(-1)
        .abs()
    )
    df_supply_demand_check_ITEMLEVEL['raw_priority_reason'] = (
        'Supply Shortage: '
        + (-df_supply_demand_check_ITEMLEVEL['supply_gap']).astype(int).astype(str)
        + ' units'
    ).where(
        df_supply_demand_check_ITEMLEVEL['feasibility'] == 'INFEASIBLE',
        '' 
    )

    df_supply_demand_check_ITEMLEVEL = df_supply_demand_check_ITEMLEVEL[['item_code','raw_priority_score','raw_priority_reason']]
    # detail_demand

    detail_demand['delivery_date'] = pd.to_datetime(detail_demand['delivery_date']).dt.date
    detail_demand = detail_demand[['item_code','delivery_date','required_qty','days_until_demand']]
    detail_demand = detail_demand.rename(columns={'days_until_demand': 'required_days'})
    #--------------------------
    
    
    # ------------ STEP 1.5: ????????????????? ------------
    
    result = detail_demand
    
    # Expand demand to all possible vendor combinations for each item
    result['vendor'] = result['item_code'].map(J_m)
    result = result.explode('vendor', ignore_index=False)
    
    # Expand to all possible brackets for each item-vendor pair
    result['bracket_no'] = list(zip(result['item_code'], result['vendor']))
    result['bracket_no'] = result['bracket_no'].map(IJ_b)
    result = result.explode('bracket_no', ignore_index=False)
    
    
    # Merge leadtime and capacity data for each item-vendor-bracket combination
    result = (
        result
        .merge(
            leadtime,
            on = ['item_code','vendor','bracket_no'],
            how ='left'
        )
        .merge(
            capacity,
            on = ['item_code','vendor'],
            how='left'
        )
    )
    
    result.loc[result['upper_bound'] == BIG_M, 'upper_bound'] = result['capacity'] #tempt handling
    
    

    # ------------ STEP 2: ITEM-DATE-VENDOR-BRACKET LEVEL ------------
    # Create bracket info string showing range and lead time
    result['bracket_info'] = (
            "'" 
            + result['lower_bound'].astype(str)
            + "-"
            + result['upper_bound'].astype(str)
            + "': "
            + result['lead_time'].astype(str)
            + " days"
    )
    
    # Check for quantity violations (below min or above max)
    qty_low_fail  = result['required_qty'] < result['lower_bound']
    qty_high_fail = result['required_qty'] > result['upper_bound']
    # Check for lead time violations
    lt_fail       = result['lead_time'] > result['required_days']
    
    # Generate demand-related error messages
    demand_msg = np.where(
        qty_low_fail,
        "[INVALID_DEMAND: need " + result['required_qty'].astype(str) + ", min = " + result['lower_bound'].astype(str) +"]",
        np.where(
            qty_high_fail,
            "[INVALID_DEMAND: need " + result['required_qty'].astype(str) + ", max = " + result['upper_bound'].astype(str) + "]",
            ""
        )
    )
    # Generate lead time error messages
    lt_msg = np.where(
        lt_fail,
        "[INVALID_LEADTIME: need <= " + result['required_days'].astype(str) + "]",
        ""
    )
    # Combine messages
    combined_msg = np.where(
        (demand_msg != "") & (lt_msg != ""),
        demand_msg + ", " + lt_msg,
        demand_msg + lt_msg
    )
    # Final comment: bracket info + validation status
    result['comment'] = np.where(
        combined_msg == "",
        "[VALID]",
        combined_msg
    )
    
    result['comment'] = result['bracket_info'] + " " + result['comment']
    
    # Row-level feasibility: True if no violations for this bracket
    result['feasible_flag'] = np.where( #row-level at grain : (item_code, delivery_date, vendor, bracket_no)
        combined_msg == "",
        True,
        False
    )
    
    # ------------ STEP 3: ITEM-DATE-VENDOR LEVEL ------------
    # Vendor-level feasibility: True if at least one bracket is feasible for this vendor
    result['vendor_feasible'] = ( #grain :(item_code,date,vendor)
        result.groupby(['item_code', 'delivery_date', 'vendor'])['feasible_flag']
        .transform('any')
    )
    

    # Aggregate to vendor level
    keys = ['item_code', 'delivery_date', 'vendor']

    result = (
        result.groupby(keys, as_index=False)
        .agg({
            'required_qty': 'first',
            'required_days': 'first',
            'vendor': 'first',  # redundant but safe
            'capacity':'max',
            'vendor_feasible': 'first',  # from previous step
            
            # combine comments
            'comment': lambda x: ' | '.join(x)
        })
        .rename(columns={'comment': 'vendor_comment'})
    )
    
    # ------------ STEP 4: ITEM-DATE LEVEL ------------
    keys = ['item_code', 'delivery_date']
    result['item_date_feasible'] = (
        result.groupby(keys)['vendor_feasible']
        .transform('any')
    )
    
    result['num_vendor_feasible'] = (
        result.groupby(keys)['vendor_feasible']
        .transform('sum')
    )
    
    # Merge with item-level priority data
    result = result.merge(df_supply_demand_check_ITEMLEVEL, on='item_code', how='left')
    
    # Calculate priority scores 
    
    cond_shortage = result['raw_priority_score'] > 0
    n = result['num_vendor_feasible']
    result['priority_score'] = np.select(
        [
            cond_shortage & (n == 0),  # Critical: shortage with no feasible vendors
            cond_shortage & (n == 1),  # High: shortage with single vendor
            cond_shortage & (n > 1)    # Medium: shortage with multiple vendors
        ],
        [
            result['raw_priority_score'] * 5,
            result['raw_priority_score'] * 2,
            result['raw_priority_score']
        ],
        default=result['raw_priority_score']  # No shortage
    )
    
    result['priority_reason'] = np.select(
        [
            cond_shortage & (n == 0),
            cond_shortage & (n == 1),
            cond_shortage & (n > 1)
        ],
        [
            result['raw_priority_reason'] + ', no valid vendor',
            result['raw_priority_reason'] + ', single vendor',
            result['raw_priority_reason']
        ],
        default=result['raw_priority_reason']
    )

    # Assign priority levels
    def _assign_priority_level(row):
        score = row['priority_score']
        vendors = row['num_vendor_feasible']
        if score == 0:
            return 'NONE'
        # base level from score
        if score <= 50:
            level = 1  # MEDIUM
        elif score <= 150:
            level = 2  # HIGH
        else:
            level = 3  # CRITICAL
        
        # escalation based on vendor risk
        if vendors == 0:
            level += 2
        elif vendors == 1:
            level += 1
        
        
        level = min(level, 3) # cap at CRITICAL
        
        return ['NONE', 'MEDIUM', 'HIGH', 'CRITICAL'][level]

    result['priority_level'] = result.apply(_assign_priority_level, axis=1)
    
    
    # Generate diagnostic messages ---
    
    result['diagnose'] = np.select(
        [
            result['priority_score'] == 0,  # No issues
            result['item_date_feasible'] == False,  # No feasible supply options
            (result['item_date_feasible'] == True) & (result['num_vendor_feasible'] <= 1),  # Feasible but risky
            result['item_date_feasible'] == True  # Fully feasible
        ],
        [   'OK: Không có vấn đề cung ứng',
            'CRITICAL: Không có phương án cung ứng hợp lệ (vi phạm min/max/leadtime)',
            'WARNING: Có khả năng cung ứng nhưng rủi ro cao (ít vendor hoặc hạn chế capacity)',
            'INFO: Có tier hợp lệ (leadtime, min, max), cần kiểm tra constraint khác'
        ]
    )
    
    # ------------ STEP 5: FINAL FORMATTING ------------
    columns = ['item_code', 'delivery_date', 'required_qty', 'required_days',
               'priority_level','priority_score', 'priority_reason',  
               'vendor', 'capacity', 'vendor_comment',
                'diagnose']
    result = result[columns]
    
    # Suppress repeated values for readability in output
    keep_data_columns = ['item_code', 'delivery_date',
                        'vendor', 'capacity', 'vendor_comment']
    keys = ['item_code', 'delivery_date']
    suppress_cols = [col for col in result.columns if col not in keep_data_columns] #columns to suppress (auto-derived)   
    mask_first = (
        result.groupby(keys)
        .cumcount() == 0
    )
    
    for col in suppress_cols: #or result[suppress_cols].where(mask_first, '')
        result[col] = np.where(mask_first, result[col], '')
        
        

    return result

#### Sheet 03_Revenue_ANALYSIS

In [14]:
import numpy as np
def analyze_revenue_vendor(df_buy_details,raw_supply):
    """
    Analyzes the revenue gap for infeasible models.
    INPUT:
    - df_buy_details: Solution data from the MaxFeasible model (contains qty purchased)
    - raw_supply: Raw supply data
    """
    
    # ---- extract input
    
    revenue = raw_supply['revenue'].copy()
    
    df_buy_details_tempt = df_buy_details.copy()
    # ------------------------
    df_revenue_by_vendor = (
        df_buy_details_tempt
        .fillna({'total_price': 0, 'qty': 0}) #handling missing values
        .groupby('vendor', as_index=False)
        .agg(
            revenue_allocated=('total_price', 'sum'),
        )
    )

    df_revenueCommit_check = (
        revenue
        .rename(columns = {'revenue': "revenue_commitment"})
        .merge(
            df_revenue_by_vendor,
            on = 'vendor',
            how = 'left'
        )
        .fillna({'revenue_commitment': 0, 'revenue_allocated': 0}) #handling missing values
        
    )

    df_revenueCommit_check['gap'] = df_revenueCommit_check['revenue_allocated'] - df_revenueCommit_check['revenue_commitment']


    df_revenueCommit_check['gap_percentage'] = np.where(
        df_revenueCommit_check['revenue_commitment'] == 0,
        np.where(df_revenueCommit_check['gap'] == 0, 0, 0),  
        (df_revenueCommit_check['gap'] / df_revenueCommit_check['revenue_commitment']) * 100
    )

    df_revenueCommit_check['gap_percentage'] = df_revenueCommit_check['gap_percentage'].round(2)


    
    df_revenueCommit_check['feasibility'] = np.where(
        df_revenueCommit_check['gap'] >= 0,
        'FEASIBLE',
        'INFEASIBLE'
    )
    conditions = [
        df_revenueCommit_check['gap_percentage'] >= 0,
        (df_revenueCommit_check['gap_percentage'] >= -5) & (df_revenueCommit_check['gap_percentage'] < 0),
        (df_revenueCommit_check['gap_percentage'] >= -20) & (df_revenueCommit_check['gap_percentage'] < -5),
        df_revenueCommit_check['gap_percentage'] < -20
    ]

    choices = [
        'OK',
        'LOW',
        'HIGH',
        'CRITICAL'
    ]

    df_revenueCommit_check['severity'] = np.select(conditions, choices, default='UNKNOWN')
    
    return df_revenueCommit_check


#### sheet 04_Relaxation Options

In [15]:
def build_relaxation_options(model, input_model):

    import pandas as pd
    # ----extract input
    

    D    = input_model["demand_data"]
    MD   = input_model["minRequiredDate_data"]
    C    = input_model["capacity_data"]
    L    = input_model["leadtime_data"]
    R    = input_model["revenue_data"]

    
    # ------------------------------
    def _get_percentage_msg(change, base):
        if base == 0: return ""
        return f"({(change / base) * 100:.2f}%)"

    # Định nghĩa các cột cố định cho Business User (Target,Actual_Plan,Adjustment_Needed)
    COLUMNS = [ 
        "Phân_Loại", "Sản_Phẩm", "Nhà_Cung_Cấp", "Khung_Giá_Bracket", 
        "Cam_Kết_Gốc", "Kế_Hoạch_Thực_Tế", "Mức_Cần_Điều_Chỉnh", "Ghi_Chú"
    ]

    rows = []

    for v in model.getVars():
        if v.X <= 1e-6 or "Art" not in v.VarName:
            continue
        
        parts = v.VarName.split('_')
        name = v.VarName
        adjustment = v.X 

        # Khởi tạo row với giá trị mặc định là "-" hoặc None để đảm bảo đủ schema
        row = {col: "-" for col in COLUMNS}

        # --- CASE 1: THỜI GIAN GIAO HÀNG (LEAD TIME) ---
        if "leadTimeFeasible" in name:
            m_code, s_code, b_no = parts[2], f"{parts[3]}_{parts[4]}", parts[5]
            target_lt = MD.get(m_code, 0)
            actual_lt = L.get((m_code, s_code, int(b_no)), 0)
            
            row.update({
                "Phân_Loại": "LEADTIME_RELAXATION",
                "Sản_Phẩm": m_code, "Nhà_Cung_Cấp": s_code, "Khung_Giá_Bracket": b_no,
                "Cam_Kết_Gốc": f"{target_lt} ngày",
                "Kế_Hoạch_Thực_Tế": f"{actual_lt} ngày",
                "Mức_Cần_Điều_Chỉnh": f"Trễ thêm {actual_lt - target_lt} ngày",
                "Ghi_Chú": "Cần nới lỏng thời gian giao hàng"
            })

        # --- CASE 2: CAM KẾT DOANH THU (REVENUE) ---
        elif "revenue" in name:
            v_code = f"{parts[-2]}_{parts[-1]}"
            target_rev = R.get(v_code, 0)
            actual_rev = target_rev - adjustment
            msg = _get_percentage_msg(adjustment, target_rev)
            
            row.update({
                "Phân_Loại": "REVENUE_REDUCTION",
                "Nhà_Cung_Cấp": v_code,
                "Cam_Kết_Gốc": f"{target_rev:,.0f}",
                "Kế_Hoạch_Thực_Tế": f"{actual_rev:,.0f}",
                "Mức_Cần_Điều_Chỉnh": f"Giảm {adjustment:,.0f} {msg}",
                "Ghi_Chú": "Không đạt doanh thu tối thiểu"
            })

        # --- CASE 3: NĂNG LỰC CUNG ỨNG (CAPACITY) ---
        elif "capacity" in name:
            m_code, s_code = parts[2], f"{parts[3]}_{parts[4]}"
            target_cap = C.get((m_code, s_code), 0)
            actual_qty = target_cap + adjustment
            msg = _get_percentage_msg(adjustment, target_cap)
            
            row.update({
                "Phân_Loại": "INCREASE_CAPACITY",
                "Sản_Phẩm": m_code, "Nhà_Cung_Cấp": s_code,
                "Cam_Kết_Gốc": f"{target_cap:,.0f}",
                "Kế_Hoạch_Thực_Tế": f"{actual_qty:,.0f}",
                "Mức_Cần_Điều_Chỉnh": f"Tăng {adjustment:,.0f} {msg}",
                "Ghi_Chú": "Cần NCC hỗ trợ sản xuất thêm"
            })

        # --- CASE 4: NHU CẦU THỊ TRƯỜNG (DEMAND) ---
        elif "demand" in name:
            m_code = parts[2]
            target_dem = D.get(m_code, 0)
            actual_dem = target_dem - adjustment
            msg = _get_percentage_msg(adjustment, target_dem)
            
            row.update({
                "Phân_Loại": "DECREASE_DEMAND",
                "Sản_Phẩm": m_code,
                "Cam_Kết_Gốc": f"{target_dem:,.0f}",
                "Kế_Hoạch_Thực_Tế": f"{actual_dem:,.0f}",
                "Mức_Cần_Điều_Chỉnh": f"Hụt {adjustment:,.0f} {msg}",
                "Ghi_Chú": "Thiếu hàng cung cấp cho thị trường"
            })

        # Nếu row đã được update thông tin (tức là thuộc 1 trong 4 case trên)
        if row["Phân_Loại"] != "-":
            rows.append(row)

    # Tạo DataFrame và ép kiểu Schema 
    result = pd.DataFrame(rows, columns=COLUMNS) #Trường hợp rows trống, df vẫn có đủ các cột đã định nghĩa
    
    
    return result

#### sheet 00_Model Summary

In [16]:
def build_model_info_RELAX(
    df_supplyDemand_check, 
    df_supplyDemand_detail_check, 
    df_revenueCommit_check,
    df_relaxation_options,
    sheet_name_list):
    

    # 01. Sheet Supply Gap
    df_infeasible_items = df_supplyDemand_check[df_supplyDemand_check['feasibility'] == "INFEASIBLE"]
    
    count_total_items = len(df_supplyDemand_check)
    count_infeasible_items = len(df_infeasible_items)
    # 02. Sheet Detail Item
    
    df_critical_items = df_supplyDemand_detail_check[df_supplyDemand_detail_check['priority_level'] == "CRITICAL" ]
    df_high_items = df_supplyDemand_detail_check[df_supplyDemand_detail_check['priority_level'] == "HIGH" ]
    
    count_total_detail_items = len(df_supplyDemand_detail_check[df_supplyDemand_detail_check['priority_level'] != "" ])
    count_critical_items = len(df_critical_items)
    count_high_items = len(df_high_items)
    # 03. Sheet Revenue
    df_infeasible_vendors = df_revenueCommit_check[df_revenueCommit_check['feasibility'] == "INFEASIBLE"]
    
    count_total_vendors = len(df_revenueCommit_check)
    count_infeasible_vendors = len(df_infeasible_vendors)
    # 04. Sheet Relax Options
    
    count_relaxation_options = len(df_relaxation_options)
    
    # PRINTING
    
    info_data = [
        ["Total Items", count_total_items, "INFO",                         sheet_name_list[0]],
        ["  - Infeasible Items (Supply Gap Violation)", count_infeasible_items, "CRITICAL", ""],
        
        ["Total Detail Items", count_total_detail_items, "INFO",        sheet_name_list[1]],
        ["  - Critical Priorities Items", count_critical_items, "CRITICAL", ""],
        ["  - High Priorities Items", count_high_items, "HIGH", ""],
        
        ["Total Vendors", count_total_vendors, "INFO",                  sheet_name_list[2]],
        ["  - Infeasible Vendors (Revenue Violation)", count_infeasible_vendors, "HIGH", ""],
        
        ["Suggested Relaxation", count_relaxation_options, "INFO",      sheet_name_list[3]]
  
    ]

    # 4. Tạo DataFrame
    df_info = pd.DataFrame(info_data, columns=["Metric", "Value","Status","Source"])
    
    return df_info



## 04. export result (to excel)

In [17]:

# feasibility

import pandas as pd
from src import config
import os
OUTPUT_FOLDER = config.output_folder
SHEET_NAME_INFEASIBLITY = config.SHEET_NAME_INFEASIBLITY


def export_result_to_excel_FEASIBILITY(solutions, constraints, model_info):
    
    # --------- EXTRACT AND PREPARE -----------------
    SHEET_NAME_FEASBILITY = [
            "00_Model_Info",
            "01_Allocation_Details",
            "02_Allocation_Summary",
            "03_Supply_Demand_Check",
            "04_Capacity_Usage_Check",
            "05_Revenue_Check",
            ]
    
    file_name = "output_FEASIBILITY.xlsx"
    file_path = os.path.join(OUTPUT_FOLDER, file_name)
    
    df_buy_details = solutions['buy_details']
    df_buy_summary = solutions['buy_summary']
    df_buy_pivot = solutions['pivot_buy']
    
    df_supply_demand_check = constraints['supply_demand']
    df_revenue_check = constraints['revenue_commitment']
    df_capacity_check = constraints['capacity_usage']
    df_capacity_summary_check = constraints['capacity_usage_summary']

    df_model_info = model_info
    # --------- EXPORT -----------------
    with pd.ExcelWriter(file_path, engine='xlsxwriter') as writer:
        # sheet 00_Model_Info
        df_model_info.to_excel(writer,                  sheet_name=SHEET_NAME_FEASBILITY[0], index=False)
        # sheet 01_Allocation_Details
        df_buy_details.to_excel(writer,                 sheet_name=SHEET_NAME_FEASBILITY[1], index=False)
        # sheet 02_Allocation_Summary
        df_buy_summary.to_excel(writer,                 sheet_name=SHEET_NAME_FEASBILITY[2], index=False,startrow=0, startcol=0)
        start_col = df_buy_summary.shape[1] + 3 #start position of second table
        df_buy_pivot.to_excel(writer,                   sheet_name=SHEET_NAME_FEASBILITY[2], index=True, startrow=0, startcol=start_col)
        # sheet 03_Supply_Demand_Check
        df_supply_demand_check.to_excel(writer,         sheet_name=SHEET_NAME_FEASBILITY[3], index=False)
        # sheet 04_Capacity_Check
        df_capacity_check.to_excel(writer,              sheet_name=SHEET_NAME_FEASBILITY[4], index=False)
        start_col = df_capacity_check.shape[1] + 3 #start position of second table
        df_capacity_summary_check.to_excel(writer,      sheet_name=SHEET_NAME_FEASBILITY[4], index=False, startrow=0, startcol=start_col)
        # sheet 05_Revnue_Check
        df_revenue_check.to_excel(writer,               sheet_name=SHEET_NAME_FEASBILITY[5], index=False)
    
    print(f"    [+] File saved to: {os.path.basename(file_path)}")
    

def export_result_to_excel_INFEASIBILITY(sheet01_supply_demand, 
                                         sheet02_detail_item, 
                                         sheet03_revenue ,
                                         sheet04_relaxation,
                                         sheet00_info):
    # --------- EXTRACT AND PREPARE -----------------

    
    file_name = "output_INFEASIBILITY.xlsx"
    file_path = os.path.join(OUTPUT_FOLDER, file_name)
    # --------- EXPORT -----------------

    
    
    
    with pd.ExcelWriter(file_path, engine='xlsxwriter') as writer:
        sheet00_info.to_excel(writer, sheet_name=SHEET_NAME_INFEASIBLITY[4], index=False)
        
        sheet01_supply_demand.to_excel(writer, sheet_name=SHEET_NAME_INFEASIBLITY[0], index=False)
        sheet02_detail_item.to_excel(writer, sheet_name=SHEET_NAME_INFEASIBLITY[1], index=False)
        sheet03_revenue.to_excel(writer, sheet_name=SHEET_NAME_INFEASIBLITY[2], index=False)
        sheet04_relaxation.to_excel(writer, sheet_name=SHEET_NAME_INFEASIBLITY[3], index=False)
        
        
    
    print(f"    [+] File saved to: {os.path.basename(file_path)}")

# 10. Run model

In [18]:
import pandas as pd
import numpy as np
import re
import os
import sys
import pandas as pd
import numpy as np

import gurobipy as gp
from gurobipy import GRB


# ----------------------------------------
sys.path.append(os.path.abspath(".."))
from src import config


CURRENT_DIR = config.CURRENT_DIR
DEBUG_FOLDER = config.debug_folder
PRODUCTION_FILE = config.production_file
VENDOR_FOLDER = config.vendor_folder
BIG_M = config.BIG_M
START_DATE = config.START_DATE
END_DATE   = config.END_DATE
PROJECT = config.PROJECT_ROOT
SHEET_NAME_INFEASIBLITY = config.SHEET_NAME_INFEASIBLITY

START_DATE = pd.to_datetime("26/12/2025", dayfirst=True)


## main logic

In [19]:
# ---------------------- 00. Loading source data ----------------------
from src.prepare_data import load_all_vendors, load_production_file
from datetime import datetime

print_header()

print(f"\n[1/4] 📂 LOADING SOURCE DATA")
print("-" * 40)
raw_supply = load_all_vendors(VENDOR_FOLDER)
raw_demand = load_production_file(PRODUCTION_FILE)
# ---------------------- 01. Prepare input ----------------------
print(f"\n[2/4] 📂 PROCESSING INPUT DATA")
print("-" * 40)
raw_demand_grouped = processing_demand(raw_demand)
model_input = prepare_model_input(raw_demand_grouped, raw_supply)

# ---------------------- 02. Build & Solve ----------------------
print(f"\n[3/4] 🧠 SOLVING BASE MODEL")
print("-" * 40)
base_model = solve_model_main(model_input)


if base_model.status == GRB.OPTIMAL:
    print(f"[!] Status    : OPTIMAL ✅")
    print(f"[!] Message   : Model solved successfully.")
    formatted_log_path = get_gurobi_log_path("01_BaseModel")
    print(f"[!] Log       : {formatted_log_path}.")
    print(f"\n[4/4] 📊 ANALYZING & FORMATTING")
    print("-" * 40)
    solutions = build_model_solution_outputs(base_model, raw_supply, model_input)
    constraints = build_model_constraint_check(solutions['buy_details'], raw_demand_grouped, raw_supply )
    df_model_info = build_model_info(base_model)
    print(f"[OK] Formatting output ")
    print(f"[OK] Exporting output ")
    export_result_to_excel_FEASIBILITY(solutions,constraints,df_model_info)
    
    
    print(f"[i] MODEL SUMMARY")
    print(f"    - Total Cost: ${base_model.ObjVal:,.2f} ")
    
elif base_model.status == GRB.INFEASIBLE:
    print(f"[!] Status    : INFEASIBLE ❌")
    print(f"[!] Message   : Model solved successfully.")

    print(f"\n    🔍STARTING INFEASIBILITY DIAGNOSIS...")
    # with base_model.copy() as max_feasible_model: optimization ram options ---> using context manager
    #     max_feasible_model = solve_with_maxFeasible(max_feasible_model)
    
    max_feasible_model = base_model.copy()
    relax_model = base_model.copy()
    
    print(f"\n    Step A: Running MaxFeasible (Resource Limit Check)")
    max_feasible_model = solve_with_maxFeasible(max_feasible_model)
    print(f"\n    Step B: Running Relaxation (Constraint Loosening)")
    relax_model = solve_with_relaxation(relax_model,model_input)
    
    
    print(f"\n[4/4] 📊 ANALYZING & FORMATTING")
    print("-" * 40)
    
    # FORMATTING
    
    solutions_max_feasible = build_model_solution_outputs(max_feasible_model, raw_supply, model_input)
    df_buy_details_max_feasible = solutions_max_feasible['buy_details']
    #---
    
    df_supplyDemand_check_max_feasible = analyze_supply_demand_gap(df_buy_details_max_feasible, raw_demand_grouped, raw_supply)
    df_detail_items_check_max_feasible = analyze_supply_demand_gap_detail_items(
                                                                    raw_demand,model_input,
                                                                    raw_supply,df_supplyDemand_check_max_feasible)
    df_revenueCommit_check_max_feasible = analyze_revenue_vendor(df_buy_details_max_feasible,raw_supply)
    df_relaxation_options = build_relaxation_options(relax_model, model_input)
    df_info = build_model_info_RELAX(df_supplyDemand_check_max_feasible, df_detail_items_check_max_feasible,
                                     df_revenueCommit_check_max_feasible, df_relaxation_options,
                                     SHEET_NAME_INFEASIBLITY)
    print(f"\n[OK] Formatting output ")
    
    # EXPORT RESULT
    export_result_to_excel_INFEASIBILITY(sheet01_supply_demand  =   df_supplyDemand_check_max_feasible,
                                         sheet02_detail_item    =   df_detail_items_check_max_feasible,
                                         sheet03_revenue        =   df_revenueCommit_check_max_feasible,
                                         sheet04_relaxation     =   df_relaxation_options,
                                         sheet00_info           =   df_info)
    
    print(f"[OK] Exporting output ")
    
    analyze_infeasbility_result(relax_model,max_feasible_model,df_relaxation_options,df_info)
    


else:
    raise Exception(f"Model is not solved optimally and can recommend relax options. Solver status: {base_model.Status}") 
    

print_final_summary(base_model)




procurement-optimizer v2.0 | 2026-04-26 12:15:38


[1/4] 📂 LOADING SOURCE DATA
----------------------------------------
[OK]  Raw Supply   : 8 vendors identified.
       [+] vendor_1.xlsx   [DONE]
       [+] vendor_2.xlsx   [DONE]
       [+] vendor_3.xlsx   [DONE]
       [+] vendor_4.xlsx   [DONE]
       [+] vendor_5.xlsx   [DONE]
       [+] vendor_6.xlsx   [DONE]
       [+] vendor_7.xlsx   [DONE]
       [+] vendor_8.xlsx   [DONE]
[OK]  Raw Demand   : 96 records loaded.
       [+] sheet demand   [DONE]
       [+] sheet delivery [DONE]
       [+] sheet bom      [DONE]

[2/4] 📂 PROCESSING INPUT DATA
----------------------------------------
[OK] Demand Aggregation: 10 unique item-groups.
[OK] Model Input Ready:
       [+] Items    : 10
       [+] Vendors  : 8
       [+] Brackets : 4
       [+] Indices  : 194 (Sparse Pairs: 65)

[3/4] 🧠 SOLVING BASE MODEL
----------------------------------------
Set parameter Username
Set parameter LicenseID to value 2794102
Set parameter LogToConsole to 